# T39 - 成交数据
## 第六章：可视化

本章介绍成交数据的可视化方法，包括：n1. 成交金额走势图
2. 行业分布图
3. 评级分布图
4. 流动性热力图
5. 综合仪表板

## 1. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
import os

# 可视化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter
import seaborn as sns

# 数据库连接
import sqlalchemy

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('可视化')

# 设置中文字体
def setup_chinese_font():
    """设置中文字体"""
    # Windows系统常用中文字体
    chinese_fonts = [
        'Microsoft YaHei',  # 微软雅黑
        'SimHei',           # 黑体
        'SimSun',           # 宋体
        'KaiTi',            # 楷体
        'FangSong',         # 仿宋
    ]
    
    for font_name in chinese_fonts:
        try:
            plt.rcParams['font.sans-serif'] = [font_name]
            plt.rcParams['axes.unicode_minus'] = False
            # 测试字体是否可用
            fig, ax = plt.subplots(figsize=(1, 1))
            ax.text(0.5, 0.5, '测试')
            plt.close(fig)
            print(f'使用字体: {font_name}')
            return True
        except:
            continue
    
    print('警告: 未找到中文字体，图表可能显示乱码')
    return False

setup_chinese_font()

# 设置图表样式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('可视化环境配置完成！')

## 2. 加载数据

In [ ]:
def load_trade_data(days: int = 60) -> pd.DataFrame:
    """加载成交数据"""
    engine = sqlalchemy.create_engine(
        config.database.get_mysql_yq_connection_string(),
        poolclass=sqlalchemy.pool.NullPool
    )
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = f'''
        SELECT * FROM yq.cjqb 
        WHERE tradedDate >= '{start_date}' AND tradedDate <= '{end_date}'
    '''
    
    df = pd.read_sql(query, engine)
    
    # 数据预处理
    if not df.empty:
        df['tradedDate'] = pd.to_datetime(df['tradedDate'])
        df['dt'] = df['tradedDate'].dt.strftime('%Y-%m-%d')
        for col in ['tradedAmount', 'tradedPrice', 'tradedYield']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f'加载 {len(df)} 条记录')
    return df

# 加载数据
df = load_trade_data(days=60)

## 3. 成交金额走势图

In [ ]:
def plot_daily_amount_trend(df: pd.DataFrame, figsize=(14, 6)):
    """
    绘制每日成交金额走势图
    
    Args:
        df: 成交数据
        figsize: 图表尺寸
    """
    if df.empty:
        print('没有数据')
        return
    
    # 按日期聚合
    daily = df.groupby('dt').agg({
        'tradedAmount': 'sum',
        'bondCode': 'nunique'
    }).reset_index()
    
    daily.columns = ['dt', 'amount', 'bond_count']
    daily['dt'] = pd.to_datetime(daily['dt'])
    
    # 计算移动平均
    daily['ma5'] = daily['amount'].rolling(5, min_periods=1).mean()
    daily['ma20'] = daily['amount'].rolling(20, min_periods=1).mean()
    
    # 创建图表
    fig, ax1 = plt.subplots(figsize=figsize)
    
    # 绘制成交金额
    color1 = '#1f77b4'
    ax1.fill_between(daily['dt'], daily['amount'], alpha=0.3, color=color1)
    ax1.plot(daily['dt'], daily['amount'], color=color1, linewidth=1.5, label='成交金额')
    ax1.plot(daily['dt'], daily['ma5'], color='orange', linewidth=2, linestyle='--', label='5日均线')
    ax1.plot(daily['dt'], daily['ma20'], color='red', linewidth=2, linestyle='-.', label='20日均线')
    
    ax1.set_xlabel('日期')
    ax1.set_ylabel('成交金额（元）', color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)
    
    # 格式化Y轴
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.1f}亿'))
    
    # 创建第二个Y轴
    ax2 = ax1.twinx()
    color2 = '#2ca02c'
    ax2.plot(daily['dt'], daily['bond_count'], color=color2, linewidth=1.5, marker='o', markersize=3, label='成交债券数')
    ax2.set_ylabel('成交债券数', color=color2)
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # 添加图例
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    plt.title('每日成交金额走势图', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 绘制走势图
plot_daily_amount_trend(df)

## 4. 行业分布图

In [ ]:
def plot_industry_distribution(df: pd.DataFrame, top_n: int = 15, figsize=(12, 8)):
    """
    绘制行业分布图
    
    Args:
        df: 成交数据
        top_n: 显示前N个行业
        figsize: 图表尺寸
    """
    if df.empty or 'yyIndustry' not in df.columns:
        print('没有行业数据')
        return
    
    # 按行业聚合
    industry = df.groupby('yyIndustry').agg({
        'tradedAmount': 'sum',
        'bondCode': 'nunique'
    }).reset_index()
    
    industry.columns = ['industry', 'amount', 'bond_count']
    industry = industry.sort_values('amount', ascending=True).tail(top_n)
    
    # 创建图表
    fig, ax = plt.subplots(figsize=figsize)
    
    # 绘制水平条形图
    bars = ax.barh(industry['industry'], industry['amount'], color=plt.cm.viridis(np.linspace(0, 1, len(industry))))
    
    # 添加数值标签
    for bar, (_, row) in zip(bars, industry.iterrows()):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width/1e8:.1f}亿', 
                ha='left', va='center', fontsize=9)
    
    ax.set_xlabel('成交金额（元）')
    ax.set_ylabel('行业')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.0f}亿'))
    
    plt.title(f'行业成交金额分布 TOP{top_n}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制行业分布图
plot_industry_distribution(df)

## 5. 评级分布图

In [ ]:
def plot_rating_distribution(df: pd.DataFrame, figsize=(10, 8)):
    """
    绘制评级分布饼图
    
    Args:
        df: 成交数据
        figsize: 图表尺寸
    """
    if df.empty or 'yyRating' not in df.columns:
        print('没有评级数据')
        return
    
    # 按评级聚合
    rating = df.groupby('yyRating').agg({
        'tradedAmount': 'sum'
    }).reset_index()
    
    rating.columns = ['rating', 'amount']
    rating = rating.sort_values('amount', ascending=False)
    
    # 合并小额项目
    total = rating['amount'].sum()
    rating['pct'] = rating['amount'] / total * 100
    
    main_ratings = rating[rating['pct'] >= 2].copy()
    other_amount = rating[rating['pct'] < 2]['amount'].sum()
    
    if other_amount > 0:
        other_row = pd.DataFrame({'rating': ['其他'], 'amount': [other_amount], 'pct': [other_amount/total*100]})
        main_ratings = pd.concat([main_ratings, other_row], ignore_index=True)
    
    # 创建饼图
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(main_ratings)))
    
    wedges, texts, autotexts = ax.pie(
        main_ratings['amount'],
        labels=main_ratings['rating'],
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        explode=[0.05] * len(main_ratings)
    )
    
    plt.setp(autotexts, size=10, weight='bold')
    plt.setp(texts, size=10)
    
    ax.set_title('评级成交金额分布', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制评级分布图
plot_rating_distribution(df)

## 6. 周内交易模式图

In [ ]:
def plot_weekday_pattern(df: pd.DataFrame, figsize=(10, 6)):
    """
    绘制周内交易模式图
    
    Args:
        df: 成交数据
        figsize: 图表尺寸
    """
    if df.empty:
        print('没有数据')
        return
    
    # 添加星期信息
    df = df.copy()
    df['weekday'] = pd.to_datetime(df['dt']).dt.dayofweek
    
    # 按星期聚合
    weekday_stats = df.groupby('weekday').agg({
        'tradedAmount': ['sum', 'mean', 'count'],
        'bondCode': 'nunique'
    })
    
    weekday_stats.columns = ['total', 'avg', 'count', 'bonds']
    weekday_stats = weekday_stats.reset_index()
    
    weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
    weekday_stats['name'] = weekday_stats['weekday'].apply(lambda x: weekday_names[x])
    
    # 只取工作日
    weekday_stats = weekday_stats[weekday_stats['weekday'] < 5]
    
    # 创建图表
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # 左图：平均成交金额
    colors = plt.cm.Blues(np.linspace(0.4, 0.8, len(weekday_stats)))
    bars = axes[0].bar(weekday_stats['name'], weekday_stats['avg'], color=colors)
    axes[0].set_ylabel('平均成交金额（元）')
    axes[0].set_title('周内平均成交金额')
    axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.1f}亿'))
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, height, 
                     f'{height/1e8:.1f}亿', 
                     ha='center', va='bottom', fontsize=9)
    
    # 右图：交易笔数
    colors = plt.cm.Oranges(np.linspace(0.4, 0.8, len(weekday_stats)))
    bars = axes[1].bar(weekday_stats['name'], weekday_stats['count'], color=colors)
    axes[1].set_ylabel('交易笔数')
    axes[1].set_title('周内交易笔数')
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, height, 
                     f'{int(height):,}', 
                     ha='center', va='bottom', fontsize=9)
    
    plt.suptitle('周内交易模式分析', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# 绘制周内交易模式图
plot_weekday_pattern(df)

## 7. 成交金额分布直方图

In [ ]:
def plot_amount_histogram(df: pd.DataFrame, figsize=(12, 6)):
    """
    绘制成交金额分布直方图
    
    Args:
        df: 成交数据
        figsize: 图表尺寸
    """
    if df.empty or 'tradedAmount' not in df.columns:
        print('没有金额数据')
        return
    
    # 过滤异常值
    amounts = df['tradedAmount'].dropna()
    amounts = amounts[(amounts > 0) & (amounts < amounts.quantile(0.99))]
    
    # 创建图表
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # 左图：线性刻度
    axes[0].hist(amounts, bins=50, color='#3498db', edgecolor='white', alpha=0.7)
    axes[0].set_xlabel('成交金额（元）')
    axes[0].set_ylabel('频数')
    axes[0].set_title('成交金额分布（线性刻度）')
    axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e4:.0f}万'))
    
    # 右图：对数刻度
    axes[1].hist(np.log10(amounts[amounts > 0]), bins=50, color='#e74c3c', edgecolor='white', alpha=0.7)
    axes[1].set_xlabel('log10(成交金额)')
    axes[1].set_ylabel('频数')
    axes[1].set_title('成交金额分布（对数刻度）')
    
    # 添加统计信息
    stats_text = f'均值: {amounts.mean()/1e4:.1f}万\n中位数: {amounts.median()/1e4:.1f}万\n标准差: {amounts.std()/1e4:.1f}万'
    axes[0].text(0.95, 0.95, stats_text, transform=axes[0].transAxes, 
                 fontsize=10, verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('成交金额分布分析', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# 绘制成交金额分布直方图
plot_amount_histogram(df)

## 8. 综合仪表板

In [ ]:
def create_dashboard(df: pd.DataFrame, figsize=(16, 12)):
    """
    创建综合仪表板
    
    Args:
        df: 成交数据
        figsize: 图表尺寸
    """
    if df.empty:
        print('没有数据')
        return
    
    # 创建子图
    fig = plt.figure(figsize=figsize)
    
    # 设置网格
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. 总体概览（文本框）
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.axis('off')
    
    total_amount = df['tradedAmount'].sum()
    total_count = len(df)
    total_bonds = df['bondCode'].nunique()
    total_days = df['dt'].nunique()
    
    summary_text = f'''
    成交数据概览
    ----------------
    总成交金额: {total_amount/1e8:.2f}亿
    总成交笔数: {total_count:,}
    成交债券数: {total_bonds:,}
    交易日数: {total_days}
    '''
    ax1.text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    # 2. 每日成交金额趋势
    ax2 = fig.add_subplot(gs[0, 1:])
    daily = df.groupby('dt')['tradedAmount'].sum().reset_index()
    daily['dt'] = pd.to_datetime(daily['dt'])
    ax2.fill_between(daily['dt'], daily['tradedAmount'], alpha=0.3, color='blue')
    ax2.plot(daily['dt'], daily['tradedAmount'], color='blue', linewidth=1.5)
    ax2.set_title('每日成交金额趋势')
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.0f}亿'))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    
    # 3. 行业分布（前10）
    ax3 = fig.add_subplot(gs[1, 0])
    if 'yyIndustry' in df.columns:
        industry = df.groupby('yyIndustry')['tradedAmount'].sum().sort_values(ascending=True).tail(10)
        industry.plot(kind='barh', ax=ax3, color='steelblue')
        ax3.set_title('行业分布TOP10')
        ax3.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.0f}亿'))
    
    # 4. 评级分布
    ax4 = fig.add_subplot(gs[1, 1])
    if 'yyRating' in df.columns:
        rating = df.groupby('yyRating')['tradedAmount'].sum().sort_values(ascending=False)
        main_ratings = rating.head(8)
        main_ratings.plot(kind='pie', ax=ax4, autopct='%1.1f%%', startangle=90)
        ax4.set_title('评级分布')
        ax4.set_ylabel('')
    
    # 5. 周内模式
    ax5 = fig.add_subplot(gs[1, 2])
    df_temp = df.copy()
    df_temp['weekday'] = pd.to_datetime(df_temp['dt']).dt.dayofweek
    weekday_stats = df_temp[df_temp['weekday'] < 5].groupby('weekday')['tradedAmount'].mean()
    weekday_names = ['周一', '周二', '周三', '周四', '周五']
    ax5.bar(weekday_names, weekday_stats.values, color='coral')
    ax5.set_title('周内平均成交金额')
    ax5.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:.1f}亿'))
    
    # 6. 成交金额分布
    ax6 = fig.add_subplot(gs[2, 0])
    amounts = df['tradedAmount'].dropna()
    amounts = amounts[(amounts > 0) & (amounts < amounts.quantile(0.95))]
    ax6.hist(amounts, bins=30, color='teal', edgecolor='white', alpha=0.7)
    ax6.set_title('成交金额分布')
    ax6.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e4:.0f}万'))
    
    # 7. 收益率分布
    ax7 = fig.add_subplot(gs[2, 1])
    if 'tradedYield' in df.columns:
        yields = df['tradedYield'].dropna()
        yields = yields[(yields > 0) & (yields < 20)]
        ax7.hist(yields, bins=30, color='purple', edgecolor='white', alpha=0.7)
        ax7.set_title('收益率分布')
        ax7.set_xlabel('收益率(%)')
    
    # 8. 每日债券数
    ax8 = fig.add_subplot(gs[2, 2])
    daily_bonds = df.groupby('dt')['bondCode'].nunique().reset_index()
    daily_bonds['dt'] = pd.to_datetime(daily_bonds['dt'])
    ax8.plot(daily_bonds['dt'], daily_bonds['bondCode'], color='green', linewidth=1.5)
    ax8.fill_between(daily_bonds['dt'], daily_bonds['bondCode'], alpha=0.3, color='green')
    ax8.set_title('每日成交债券数')
    plt.setp(ax8.xaxis.get_majorticklabels(), rotation=45)
    
    plt.suptitle('成交数据综合仪表板', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# 创建综合仪表板
create_dashboard(df)